# Capítulo 6 — Evaluar un modelo y medir la incertidumbre · Solución comentada

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch06_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y luego muestra su equivalente en Python. En cada cálculo, los comentarios indican el conjunto utilizado.

**Índice.** Datos · Los cálculos de las lecciones 6.1 a 6.6 · Ejercicios 6.1 a 6.6

**El protocolo, en todo el notebook:** ajustar en el **entrenamiento**, elegir en la **validación**, evaluar **una sola vez** en la **prueba**. Para una prueba A/B: comparar dos grupos **asignados al azar**, y dar la diferencia **con su intervalo de confianza**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color por conjunto
LOT_E, LOT_V, LOT_T = '#2e7d32', '#b84d00', '#1565c0'   # entrenamiento (verde), validación (naranja oscuro), prueba (azul)
SPAM, LEGITIME = '#d62728', '#b9b6ae'
VERSION_A, VERSION_B, ECART, SEUIL = '#757575', '#00897b', '#c2185b', '#8e24aa'   # prueba A/B: A gris, B verde azulado


def sigmoide(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def perte_moyenne(etiquettes, X, w, b):
    """Pérdida logarítmica media (ecuaciones 5.6 y 5.7)."""
    p = np.clip(sigmoide(X @ np.asarray(w, dtype=float) + b), 1e-12, 1 - 1e-12)
    y = np.asarray(etiquettes, dtype=float)
    return float(np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p))))


NOMS_CARAC = ['liens', 'mots_suspects', 'longueur', 'majuscules', 'heure', 'pieces_jointes', 'taille', 'destinataires']


def fabriquer_messages(graine=77):
    """Mensajes ficticios (ejercicio 6.4 y lección 6.4): 40 grupos, campañas de spam (grupos pares) o
    remitentes legítimos (grupos impares). Los mensajes de un grupo son casi idénticos. Solo las dos
    primeras características dependen de la etiqueta; las ocho están en unidades muy distintas."""
    rng = np.random.default_rng(graine)
    unites = np.array([2, 1.5, 30, 0.2, 5, 1, 50, 3])
    decalages = np.array([3, 2, 200, 1, 12, 0, 500, 10])
    lignes = []
    for g in range(40):
        spam = g % 2 == 0
        taille = rng.integers(4, 15)
        base = rng.normal(0, 1, 8)
        base[:2] += 1.0 if spam else -1.0
        for _ in range(taille):
            lignes.append([g, int(spam)] + list((base + rng.normal(0, 0.3, 8)) * unites + decalages))
    return pd.DataFrame(lignes, columns=['groupe', 'spam'] + NOMS_CARAC)


def ajuster(X, y, lam, l1=False, pas=0.1, n_pas=3000):
    """Descenso de gradiente sobre la pérdida penalizada: L2 (ecuación 6.2) o L1 (ecuación 6.3). b no se penaliza."""
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(n_pas):
        ecart = sigmoide(X @ w + b) - y
        gw, gb = X.T @ ecart / len(y), ecart.mean()
        if l1:   # paso de gradiente, luego acercamos cada peso a 0 como mucho en pas × λ (sin cambiarle el signo)
            w = w - pas * gw
            w = np.sign(w) * np.maximum(np.abs(w) - pas * lam, 0)
        else:    # la derivada de λ × suma de los w² es 2λw
            w = w - pas * (gw + 2 * lam * w)
        b -= pas * gb
    return w, b

## Datos

Los identificadores D6.1 a D6.5, D6.E1, D6.E2, D6.E3, D6.E5 y D5.1 son los del libro y de los archivos CSV.

Los nombres de variables y de columnas del código se mantienen en francés: `lot` (conjunto: `entrainement` entrenamiento, `validation` validación, `test` prueba), `seuil` (umbral), `fn` / `fp` (falsos negativos / falsos positivos), `coût` (coste), `pli` (pliegue), `perte` (pérdida), `fuite` (fuga), `decoupage` (división), `groupe` (grupo), `etiquette` (etiqueta), `liens` (enlaces), `mots_suspects` (palabras sospechosas), `mediatheque` (mediateca), `librairie` (librería), `clients` / `commandes` (clientes / pedidos), `spectateurs` / `reservations` (espectadores / reservas).

In [ ]:
# Datos del capítulo 6, copiados de los archivos CSV del libro (no modificar)

# D6.1 — los tres conjuntos de la asociación: mensajes, spams y errores del filtro para un umbral
lots = pd.DataFrame([
    ('entrainement', 600, 120, 0.7, 30, 3),
    ('validation', 200, 40, 0.3, 4, 14),
    ('validation', 200, 40, 0.5, 8, 6),
    ('validation', 200, 40, 0.7, 15, 2),
    ('test', 250, 50, 0.7, 20, 4)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.2 — dos divisiones de los mismos mensajes: el conjunto de prueba, con umbral 0,7
fuite = pd.DataFrame([
    ('aleatoire', 250, 50, 25, 1, 10, 4),
    ('par_campagne', 250, 50, 0, 0, 20, 4)
], columns=['decoupage', 'messages', 'spams', 'copies_vues', 'manques_copies', 'manques_nouveaux', 'fp'])

# D6.3 — seis mensajes nuevos (validación de la lección 6.3)
six = pd.DataFrame([
    ('R1', 'spam', 4, 2),
    ('R2', 'legitime', 1, 3),
    ('R3', 'spam', 3, 4),
    ('R4', 'legitime', 0, 0),
    ('R5', 'spam', 5, 3),
    ('R6', 'legitime', 2, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D5.1 — los doce mensajes del capítulo 5 (entrenamiento de la lección 6.3)
douze = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'etiquette', 'liens', 'mots_suspects'])

# D6.E2 — la mediateca (ejercicio 6.2): conjuntos de validación y de prueba
mediatheque = pd.DataFrame([
    ('validation', 150, 30, 0.4, 3, 18),
    ('validation', 150, 30, 0.5, 6, 9),
    ('validation', 150, 30, 0.6, 10, 5),
    ('validation', 150, 30, 0.7, 15, 3),
    ('test', 240, 48, 0.4, 6, 26),
    ('test', 240, 48, 0.5, 10, 14),
    ('test', 240, 48, 0.6, 17, 9),
    ('test', 240, 48, 0.7, 22, 3)
], columns=['lot', 'messages', 'spams', 'seuil', 'fn', 'fp'])

# D6.4 — los tres pliegues de la asociación (lección 6.4): enlaces (media, desviación típica) y pérdidas de validación
plis = pd.DataFrame([
    (1, 270, 54, 2.10, 1.60, 0.29, 0.30),
    (2, 264, 52, 2.40, 1.90, 0.38, 0.33),
    (3, 266, 54, 2.20, 1.70, 0.33, 0.31)
], columns=['pli', 'messages', 'spams', 'moyenne_liens', 'ecart_type_liens', 'perte_l001', 'perte_l01'])

# D6.5 — la prueba A/B de la librería (lecciones 6.5 y 6.6)
librairie = pd.DataFrame([
    ('A', 'lettre actuelle', 2500, 200),
    ('B', 'livraison offerte', 2400, 240)
], columns=['version', 'description', 'clients', 'commandes'])

# D6.E1 — validación cruzada de la mediateca (ejercicio 6.1): pérdida de cada pliegue para cinco λ
plis_lambda = pd.DataFrame([
    (1, 0.001, 0.52),
    (1, 0.01, 0.44),
    (1, 0.1, 0.36),
    (1, 1, 0.35),
    (1, 10, 0.52),
    (2, 0.001, 0.40),
    (2, 0.01, 0.37),
    (2, 0.1, 0.34),
    (2, 1, 0.38),
    (2, 10, 0.55),
    (3, 0.001, 0.61),
    (3, 0.01, 0.49),
    (3, 0.1, 0.39),
    (3, 1, 0.37),
    (3, 10, 0.51)
], columns=['pli', 'lambda', 'perte'])

# D6.E3 — reparto de cuatro grupos en los pliegues del voluntario (ejercicio 6.3)
groupes_plis = pd.DataFrame([
    ('Montres', 'campagne de spams', 9, 8, 9),
    ('Colis en attente', 'campagne de spams', 5, 6, 4),
    ('Loterie', 'campagne de spams', 7, 7, 8),
    ('Lettre de la mairie', 'expéditeur légitime', 4, 3, 5)
], columns=['groupe', 'nature', 'pli_1', 'pli_2', 'pli_3'])

# D6.E5 — la prueba A/B del teatro (ejercicio 6.5)
theatre = pd.DataFrame([
    ('A', 'lettre habituelle', 1800, 126),
    ('B', 'lettre avec vidéo', 1750, 147)
], columns=['version', 'description', 'spectateurs', 'reservations'])

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 6.1 a 6.6.

In [ ]:
# Lección 6.1 — coste por mensaje (ecuación 6.1), k = 3 para la asociación
k = 3
t = lots.copy()
t['coût'] = t['fn'] + k * t['fp']
t['coût par message'] = (t['coût'] / t['messages']).round(3)    # se divide por el tamaño DEL CONJUNTO
print(t.to_string(index=False))
val = t[t['lot'] == 'validation']
choisi = val.loc[val['coût par message'].idxmin(), 'seuil']
print('umbral elegido en la validación:', choisi)
test = t[(t['lot'] == 'test') & (t['seuil'] == choisi)].iloc[0]
print(f"prueba, una sola vez: coste {test['coût']} sobre {test['messages']} mensajes = {test['coût par message']:.3f} por mensaje")

a7 = t[t['seuil'] == 0.7]
couleurs = {'entrainement': LOT_E, 'validation': LOT_V, 'test': LOT_T}
plt.figure(figsize=(5, 2.8))
plt.bar(a7['lot'], a7['coût par message'], color=[couleurs[l] for l in a7['lot']])
plt.ylabel('Coste por mensaje (umbral 0,7)'); plt.show()

In [ ]:
# Lección 6.2 — una fuga de datos: exhaustividad (recall) y coste por mensaje de las dos divisiones
for _, l in fuite.iterrows():
    manques = l['manques_copies'] + l['manques_nouveaux']
    nouveaux = l['spams'] - l['copies_vues']
    print(f"{l['decoupage']:>12} : recall = {(l['spams'] - manques) / l['spams']:.2f}, "
          f"coste por mensaje = {(manques + 3 * l['fp']) / l['messages']:.3f}, "
          f"parte de spams nuevos no detectados = {l['manques_nouveaux']} / {nouveaux} = {l['manques_nouveaux'] / nouveaux:.2f}")

# Dividir las campañas, no los mensajes: pequeña demostración con 40 campañas de tamaño variable
rng = np.random.default_rng(6)
campagne = np.repeat(np.arange(40), rng.integers(5, 40, size=40))          # número de campaña de cada spam
au_hasard = rng.random(campagne.size) < 0.25                              # 25 % de los MENSAJES en la prueba
camp_test = rng.choice(40, size=10, replace=False)                         # 25 % de las CAMPAÑAS en la prueba
par_campagne = np.isin(campagne, camp_test)
for nom, test_ in (('al azar', au_hasard), ('por campaña', par_campagne)):
    partagees = len(set(campagne[test_]) & set(campagne[~test_]))
    print(f'{nom:>12} : {test_.sum()} spams en la prueba, {partagees} campañas presentes a la vez en la prueba y en otra parte')

In [ ]:
# Lección 6.3 — sobreajuste: filtros A y B en el entrenamiento (doce mensajes) y en la validación (seis)
X12, y12 = douze[['liens', 'mots_suspects']].to_numpy(float), (douze['etiquette'] == 'spam').to_numpy(int)
X6, y6 = six[['liens', 'mots_suspects']].to_numpy(float), (six['etiquette'] == 'spam').to_numpy(int)
filtres = {'A': ((1, 0.5), -4), 'B': ((-0.54, 4.39), -9.17)}
for nom, (w, b) in filtres.items():
    p6 = sigmoide(X6 @ np.array(w) + b)
    pertes6 = -(y6 * np.log(p6) + (1 - y6) * np.log(1 - p6))
    print(f'filtro {nom}: entrenamiento {perte_moyenne(y12, X12, w, b):.2f}, validación {perte_moyenne(y6, X6, w, b):.2f} '
          f'| pérdidas de los seis: {np.round(pertes6, 2)}')

# Pérdida penalizada L2 (ecuación 6.2), λ = 0,05; la constante b no se penaliza
lam = 0.05
for nom, (w, b) in filtres.items():
    pen = lam * np.sum(np.square(w))
    print(f'filtro {nom}: suma de los cuadrados de los pesos = {np.sum(np.square(w)):.4f}, penalización = {pen:.4f}, '
          f'pérdida penalizada = {perte_moyenne(y12, X12, w, b) + pen:.2f}')

# La penalización depende de las unidades: enlaces contados en decenas, mismas predicciones, otra penalización
X12_diz = X12 / np.array([10, 1])
w_diz = (10, 0.5)
print('mismas probabilidades:', np.allclose(sigmoide(X12_diz @ np.array(w_diz) - 4), sigmoide(X12 @ np.array([1, 0.5]) - 4)),
      f'| penalización en unidades: {lam * (1 + 0.25):.4f}; en decenas: {lam * (100 + 0.25):.4f}')

In [ ]:
# Lección 6.4 — estandarización en cada vuelta: media y desviación típica de los enlaces en los OTROS DOS pliegues
for k in range(3):
    autres = plis[plis['pli'] != k + 1]
    n = autres['messages'].sum()
    m = (autres['messages'] * autres['moyenne_liens']).sum() / n
    # desviación típica de la unión de dos pliegues (división por n): media de los cuadrados menos el cuadrado de la media
    v = (autres['messages'] * (autres['ecart_type_liens'] ** 2 + autres['moyenne_liens'] ** 2)).sum() / n - m ** 2
    print(f'vuelta {k + 1}: validación = pliegue {k + 1}, entrenamiento = {n} mensajes, media {m:.2f}, desviación típica {np.sqrt(v):.2f}')
print(f'mensaje con 6 enlaces del pliegue 3, en la vuelta 3: (6 - 2,25) / 1,76 = {(6 - 2.25) / 1.76:.2f}')

# Puntuación de validación cruzada (ecuación 6.4) de los dos valores de λ
for col, lam in (('perte_l001', 0.01), ('perte_l01', 0.1)):
    print(f'λ = {lam}: pérdidas de los pliegues {list(plis[col])}, VC = {plis[col].mean():.3f}')

In [ ]:
# Lección 6.4 (final) — L1 y L2 sobre mensajes ficticios con ocho características (los mismos que en el ejercicio 6.4)
msgs = fabriquer_messages()
X8 = msgs[NOMS_CARAC].to_numpy(float)
X8 = (X8 - X8.mean(0)) / X8.std(0)        # aquí, estandarización sobre todos los mensajes: simple demostración
y8 = msgs['spam'].to_numpy(float)
w_l2, _ = ajuster(X8, y8, 0.1)
w_l1, _ = ajuster(X8, y8, 0.1, l1=True)
print('pesos L2:', np.round(w_l2, 2))
print('pesos L1:', np.round(w_l1, 2))
print(f'pesos nulos: L2 {int(np.sum(np.abs(w_l2) < 1e-9))}, L1 {int(np.sum(np.abs(w_l1) < 1e-9))} '
      '(las dos primeras características son las únicas relacionadas con la etiqueta)')

In [ ]:
# Lecciones 6.5 y 6.6 — la prueba A/B de la librería
a, b_ = librairie.iloc[0], librairie.iloc[1]
pA, pB = a['commandes'] / a['clients'], b_['commandes'] / b_['clients']
ecart = pB - pA
print(f'tasa A = {100 * pA:.1f} %, tasa B = {100 * pB:.1f} %, diferencia = {100 * ecart:.1f} puntos, '
      f'variación relativa = {100 * ecart / pA:.0f} %')
esA, esB = np.sqrt(pA * (1 - pA) / a['clients']), np.sqrt(pB * (1 - pB) / b_['clients'])
es = np.sqrt(esA ** 2 + esB ** 2)                          # ecuación 6.6: se suman los CUADRADOS
bas, haut = ecart - 1.96 * es, ecart + 1.96 * es           # ecuación 6.7
print(f'EE de A = {100 * esA:.2f} punto, EE de B = {100 * esB:.2f} punto, EE de la diferencia = {100 * es:.2f} punto')
print(f'intervalo de confianza al 95 %: de {100 * bas:.1f} a {100 * haut:.1f} puntos')
print(f'para 40 000 suscriptores: {40000 * ecart:.0f} pedidos más (de {40000 * bas:.0f} a {40000 * haut:.0f})')

plt.figure(figsize=(6, 1.8))
plt.axvline(0, color='black', linewidth=2.5, label='0: ningún efecto')
plt.axvline(1, color=SEUIL, linestyle='--', label='umbral de rentabilidad: 1 punto')
plt.plot([100 * bas, 100 * haut], [0, 0], color=ECART, linewidth=4, label='intervalo al 95 %')
plt.scatter([100 * ecart], [0], marker='D', s=70, color=ECART, zorder=3)
plt.yticks([]); plt.xlim(-1, 5); plt.xlabel('Efecto de la versión B (puntos porcentuales)')
plt.legend(fontsize=7, loc='upper right'); plt.show()

# Lección 6.6 — el coste por mensaje de la prueba (lección 6.1) es una media: mismo razonamiento
couts = np.array([1] * 20 + [3] * 4 + [0] * 226)            # 20 spams no detectados, 4 falsas alarmas, 226 sin coste
es_c = couts.std() / np.sqrt(len(couts))                     # desviación típica (división por n) / raíz de n
print(f'coste por mensaje = {couts.mean():.3f}, desviación típica = {couts.std():.3f}, EE = {es_c:.3f}, '
      f'intervalo: de {couts.mean() - 1.96 * es_c:.2f} a {couts.mean() + 1.96 * es_c:.2f}')

## Ejercicio 6.1 — Leer un gráfico: tres pliegues, cinco valores de λ

*Conceptos: validación cruzada (lección 6.4, ecuación 6.4), sobreajuste y penalización (lección 6.3).* Datos D6.E1: la pérdida de validación de cada pliegue de la mediateca, para cinco valores de $\lambda$.

### a) a e) Solución comentada

**a)** Pliegue 2 para $\lambda = 0{,}1$: 0,34. Pliegue 3 para $\lambda = 0{,}001$: 0,61.

**b)** Las medias de los tres pliegues valen aproximadamente 0,51, 0,43, 0,36, 0,37 y 0,53. La más pequeña es la de $\lambda = 0{,}1$ (0,363). Por separado, el pliegue 1 habría elegido $\lambda = 1$ (0,35), el pliegue 2 $\lambda = 0{,}1$ (0,34) y el pliegue 3 $\lambda = 1$ (0,37): dos pliegues de cada tres prefieren un valor distinto del de la media.

**c)** Con $\lambda = 0{,}001$, la penalización casi no frena nada: el filtro aprende de memoria las particularidades de los pliegues de entrenamiento, como el filtro B de la lección 6.3, y se equivoca con seguridad en el pliegue de validación. Con $\lambda = 10$, la penalización aplasta todos los pesos hacia 0: el filtro da casi la misma probabilidad a todos los mensajes y ya no distingue los spams. Entre ambos, la pérdida pasa por un mínimo.

**d)** Las medias de $\lambda = 0{,}1$ y $\lambda = 1$ difieren en unos 0,003. Para un mismo $\lambda$, los pliegues difieren mucho más: de 0,34 a 0,39 para $\lambda = 0{,}1$. La elección entre esos dos valores no es, por tanto, sólida: a la vista de los datos, son igual de buenos. En cambio, descartar 0,001 y 10 sí es una elección sólida.

**e)** El 0,36 es la puntuación de validación cruzada que sirvió para **elegir** $\lambda$: favorece ligeramente el ajuste elegido. La mediateca debe primero ajustar el modelo final en todos sus pliegues con el $\lambda$ elegido, fijar su umbral y después medir **una sola vez** la pérdida en un conjunto de prueba que se haya mantenido aparte. Ese es el número que puede anunciar.

In [ ]:
moyennes = plis_lambda.groupby('lambda')['perte'].mean()     # VC(λ): media de los tres pliegues
lam_choisi = moyennes.idxmin()

In [ ]:
# Comprobación de a) y b)
if moyennes is None or lam_choisi is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    print('medias:', moyennes.round(3).to_dict(), '→',
          'correcto' if np.allclose(moyennes.values, [0.51, 1.3 / 3, 1.09 / 3, 1.1 / 3, 1.58 / 3]) else 'revisar')
    print('λ elegido:', lam_choisi, '→', 'correcto' if lam_choisi == 0.1 else 'revisar')
    print('elección de cada pliegue por separado:', plis_lambda.loc[plis_lambda.groupby('pli')['perte'].idxmin(), ['pli', 'lambda']].values.tolist())
    for k, (style, marque) in enumerate((('-', 'o'), ('--', 's'), (':', '^'))):
        d = plis_lambda[plis_lambda['pli'] == k + 1]
        plt.plot(d['lambda'], d['perte'], style, marker=marque, color=LOT_V, label=f'pliegue {k + 1}')
    plt.plot(moyennes.index, moyennes.values, 'D-', color='black', linewidth=2.5, label='media')
    plt.xscale('log'); plt.xlabel('λ'); plt.ylabel('Pérdida de validación'); plt.legend(); plt.grid(alpha=0.4); plt.show()

## Ejercicio 6.2 — Elegir y después evaluar un umbral: la mediateca

*Conceptos: tres conjuntos (lección 6.1), coste por mensaje (ecuación 6.1), coste de un umbral (ecuación 5.9).* Para la mediateca, una falsa alarma cuesta lo mismo que dos spams no detectados: $k = 2$. Datos D6.E2. Haz primero las preguntas a) a e) en papel, **sin mirar las filas de la prueba antes de la pregunta b)**.

### a) a e) Solución comentada

**a)** En la **validación** (150 mensajes), con $FN + 2 \times FP$:

| Umbral | Coste | Coste por mensaje |
|:--|--:|--:|
| 0,4 | $3 + 2 \times 18 = 39$ | $39 / 150 = 0{,}260$ |
| 0,5 | $6 + 2 \times 9 = 24$ | $24 / 150 = 0{,}160$ |
| 0,6 | $10 + 2 \times 5 = 20$ | $20 / 150 \approx 0{,}133$ |
| 0,7 | $15 + 2 \times 3 = 21$ | $21 / 150 = 0{,}140$ |

El umbral **0,6** es el que menos cuesta en la validación. Nos quedamos con él.

**b)** En la **prueba** (240 mensajes), solo con el umbral 0,6: $17 + 2 \times 9 = 35$, y después $35 / 240 \approx 0{,}146$ por mensaje. Es el número que hay que anunciar.

**c)** Los dos conjuntos no tienen el mismo tamaño: 150 y 240 mensajes. Los totales, 20 y 35, harían creer que el filtro cuesta un 75 % más en la prueba. Por mensaje, la diferencia es solo de 0,133 a 0,146, es decir, alrededor de un 9 % más. Un total crece con el número de mensajes; un coste por mensaje, no.

**d)** Sí. El umbral se eligió porque era el que mejor funcionaba en esos 150 mensajes concretos, con su parte de suerte. Su rendimiento en validación es, por tanto, ligeramente favorable. La prueba, que no sirvió para decidir nada, no se beneficia de esa suerte.

**e)** Primer error: **elige el umbral en la prueba**, que se convierte en una segunda validación. Segundo error: anuncia el **mejor** de los cuatro costes de la prueba, 0,117, que es forzosamente favorable: es un mínimo, no una medida. Debería anunciar 0,146 por mensaje, con el umbral 0,6 elegido en la validación. Si quiere comparar 0,6 y 0,7, necesita una validación más grande o una validación cruzada, y después un nuevo conjunto de prueba. Observa, por cierto, que en la validación 0,6 y 0,7 solo difieren en una unidad de coste sobre 150 mensajes: la elección entre ellos es frágil.

In [ ]:
def cout_par_message(fn, fp, n, k):
    return (fn + k * fp) / n                    # se divide por el tamaño del conjunto


def choisir_seuil(table_validation, k):
    """Devuelve el umbral cuyo coste por mensaje es el más pequeño en la validación."""
    couts = [cout_par_message(l.fn, l.fp, l.messages, k) for l in table_validation.itertuples()]
    return table_validation['seuil'].iloc[int(np.argmin(couts))]   # aquí no se mira la prueba

In [ ]:
# Comprobación de a) y b), y después el protocolo
k_med = 2
valid = mediatheque[mediatheque['lot'] == 'validation']
test_med = mediatheque[mediatheque['lot'] == 'test']
if cout_par_message(3, 18, 150, 2) is None or choisir_seuil(valid, k_med) is None:
    print('aún sin completar. Completa las funciones.')
else:
    cv = [cout_par_message(l.fn, l.fp, l.messages, k_med) for l in valid.itertuples()]
    print('validación, coste por mensaje:', np.round(cv, 3), '→',
          'correcto' if np.allclose(cv, [39 / 150, 24 / 150, 20 / 150, 21 / 150]) else 'revisar')
    s = choisir_seuil(valid, k_med)
    print(f'umbral elegido en la validación: {s} →', 'correcto' if s == 0.6 else 'revisar')
    l = test_med[test_med['seuil'] == s].iloc[0]                    # UNA sola fila de la prueba
    ct = cout_par_message(l.fn, l.fp, l.messages, k_med)
    print(f'prueba, solo con el umbral {s}: {ct:.3f} por mensaje →', 'correcto' if abs(ct - 35 / 240) < 1e-9 else 'revisar')
    # e) Lo que hace el colega: mira TODAS las filas de la prueba y se queda con la mejor
    ctous = [cout_par_message(r.fn, r.fp, r.messages, k_med) for r in test_med.itertuples()]
    s_coll = test_med['seuil'].iloc[int(np.argmin(ctous))]
    print(f'anuncio del colega: umbral {s_coll}, {min(ctous):.3f} por mensaje (elegido en la prueba: demasiado optimista);',
          f'anuncio honesto: umbral {s}, {ct:.3f} por mensaje →',
          'correcto' if s_coll == 0.7 and abs(min(ctous) - 28 / 240) < 1e-9 else 'revisar')
    plt.figure(figsize=(5.5, 3))
    plt.plot(valid['seuil'], cv, 'o-', color=LOT_V, label='validación (sirve para elegir)')
    plt.plot(test_med['seuil'], ctous, 's--', color=LOT_T, alpha=0.5, label='prueba (no mirarla para elegir)')
    plt.scatter([s], [ct], s=120, color=LOT_T, zorder=3, label='prueba, con el umbral elegido')
    plt.xlabel('Umbral'); plt.ylabel('Coste por mensaje (k = 2)'); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### f) Solución comentada

La mediateca puede anunciar: «Con el umbral 0,6, elegido en la validación, el filtro cuesta unos 0,146 por mensaje en un conjunto de prueba de 240 mensajes que no había servido para nada más». El gráfico muestra por qué el anuncio del colega es demasiado bonito: al elegir el punto más bajo de la curva de la prueba, convierte la prueba en validación. Su 0,117 ya no es una medida del coste futuro. Por último, 240 mensajes no dan un coste exacto: la lección 6.6 enseña a decir cuánto puede variar.

## Ejercicio 6.3 — Interpretar: una fuga entre los pliegues

*Conceptos: fuga de datos (lección 6.2), validación cruzada por grupos y estandarización en cada vuelta (lección 6.4).* Datos D6.E3: dónde cayeron los mensajes de cuatro grupos en los pliegues del voluntario.

### a) a d) Solución comentada

**a)** En la vuelta 1, los pliegues 2 y 3 forman el entrenamiento: el filtro ha visto $8 + 9 = 17$ copias de la campaña « Montres » (relojes). Encuentra 9 en el pliegue 1, en validación.

**b)** **Paso 2**: el pliegue se sortea mensaje a mensaje. Por tanto, cada campaña queda dividida entre los pliegues (los cuatro grupos de la tabla lo están). En cada vuelta, el filtro ha visto en el entrenamiento copias casi idénticas a los mensajes de validación: reconoce la campaña en lugar de juzgar un mensaje nuevo. **Paso 1**: la media y la desviación típica se calculan sobre los 800 mensajes, es decir, incluyendo los mensajes de cada pliegue de validación. Esos dos números hacen pasar un poco de información de la validación al entrenamiento. Esta segunda fuga es más pequeña, pero del mismo tipo.

**c)** La pérdida de prueba, 0,34, es la que mejor describe los mensajes futuros: el conjunto de prueba se formó por campañas y solo se usó una vez. La puntuación de validación cruzada, 0,11, es demasiado buena porque la validación contenía copias ya vistas. Los pasos 4 y 5 son correctos en sí mismos: elegir $\lambda$ por validación cruzada y después abrir la prueba una sola vez. Pero el paso 4 trabaja sobre puntuaciones falseadas: la fuga premia a los filtros que aprenden de memoria, de modo que el $\lambda$ elegido puede ser también demasiado pequeño.

**d)** Los grupos tienen 26, 15, 22 y 12 mensajes, es decir, 75; lo ideal serían 25 por pliegue. Un reparto posible: pliegue 1 = Montres (26), pliegue 2 = Loterie (22), pliegue 3 = Colis en attente y Lettre de la mairie (27). Paso 1 reescrito: «En cada vuelta, calcular la media y la desviación típica de cada característica solo en los dos pliegues de entrenamiento, y después estandarizar con esos dos números los mensajes de entrenamiento y los del pliegue de validación».

In [ ]:
def groupes_coupes(repartition):
    """repartition: una tabla con una columna 'groupe' y las columnas 'pli_1', 'pli_2', 'pli_3'
    (número de mensajes del grupo en cada pliegue). Devuelve la lista de los grupos divididos."""
    nb_plis = (repartition[['pli_1', 'pli_2', 'pli_3']] > 0).sum(axis=1)   # ¿en cuántos pliegues está el grupo?
    return list(repartition.loc[nb_plis > 1, 'groupe'])

In [ ]:
# Comprobación de b) y d)
proposition = pd.DataFrame([('Montres', 26, 0, 0), ('Colis en attente', 0, 0, 15),
                            ('Loterie', 0, 22, 0), ('Lettre de la mairie', 0, 0, 12)],
                           columns=['groupe', 'pli_1', 'pli_2', 'pli_3'])
if groupes_coupes(groupes_plis) is None:
    print('aún sin completar. Completa la función.')
else:
    print('pliegues del voluntario, grupos divididos:', groupes_coupes(groupes_plis), '→',
          'correcto' if len(groupes_coupes(groupes_plis)) == 4 else 'revisar')
    print('reparto corregido, grupos divididos:', groupes_coupes(proposition), '| tamaños de los pliegues:',
          proposition[['pli_1', 'pli_2', 'pli_3']].sum().tolist(), '→',
          'correcto' if groupes_coupes(proposition) == [] else 'revisar')
    print('vuelta 1: copias de « Montres » vistas en el entrenamiento =', int(groupes_plis.loc[0, ['pli_2', 'pli_3']].sum()),
          ', en la validación =', int(groupes_plis.loc[0, 'pli_1']))

## Ejercicio 6.4 — Validación cruzada por campañas

*Conceptos: validación cruzada (lección 6.4, ecuación 6.4), penalización L2 (ecuación 6.2), estandarización (ecuación 2.7), fuga (lección 6.2).* Los mensajes los genera `fabriquer_messages()` (celda de importación). Los grupos 30 a 39 forman el conjunto de prueba: no se tocan hasta la pregunta e).

In [ ]:
# Los mensajes del ejercicio: 30 grupos para la validación cruzada, 10 grupos apartados para la prueba
msgs = fabriquer_messages()
test_ = msgs['groupe'] >= 30
X_cv, y_cv, g_cv = msgs.loc[~test_, NOMS_CARAC].to_numpy(float), msgs.loc[~test_, 'spam'].to_numpy(float), msgs.loc[~test_, 'groupe'].to_numpy()
X_test, y_test = msgs.loc[test_, NOMS_CARAC].to_numpy(float), msgs.loc[test_, 'spam'].to_numpy(float)
print(f'{len(msgs)} mensajes, {msgs.groupe.nunique()} grupos; validación cruzada: {len(y_cv)} mensajes; prueba: {len(y_test)} mensajes')
print(msgs[NOMS_CARAC].describe().loc[['mean', 'std']].round(1))     # unidades muy distintas

### a) Solución comentada

Se espera que **baje y luego vuelva a subir**. Sin penalización, el filtro puede dar pesos grandes a las seis características sin relación con la etiqueta, que varían al azar de un grupo a otro: se sobreajusta. Con una penalización demasiado fuerte, todos los pesos quedan aplastados.

In [ ]:
def plis_par_groupe(groupes, K):
    """Número de pliegue (0 a K - 1) de cada mensaje: los grupos, ordenados de menor a mayor, se reparten
    por turnos en los pliegues; todos los mensajes de un grupo reciben el mismo pliegue."""
    rangs = {g: i % K for i, g in enumerate(np.unique(groupes))}
    return np.array([rangs[g] for g in groupes])


def standardiser(X_entr, X_val):
    """Estandariza X_entr y X_val con la media y la desviación típica de X_entr SOLAMENTE."""
    m, s = X_entr.mean(axis=0), X_entr.std(axis=0)
    return (X_entr - m) / s, (X_val - m) / s


def score_vc(lam, plis_, K=3):
    """Ecuación 6.4: media, sobre las K vueltas, de la pérdida SIN penalización en el pliegue de validación."""
    pertes = []
    for k in range(K):
        entr, val = plis_ != k, plis_ == k
        Xe, Xv = standardiser(X_cv[entr], X_cv[val])          # números calculados dentro de la vuelta, sobre el entrenamiento
        w, b = ajuster(Xe, y_cv[entr], lam)
        pertes.append(perte_moyenne(y_cv[val], Xv, w, b))     # se juzgan las predicciones, sin penalización
    return float(np.mean(pertes))

In [ ]:
# Comprobación de b) y c)
LAMBDAS = [0, 0.001, 0.01, 0.1, 1]
p_g = plis_par_groupe(g_cv, 3)
if p_g is None or standardiser(X_cv, X_cv)[0] is None or score_vc(0.1, p_g) is None:
    print('aún sin completar. Completa las tres funciones.')
else:
    coupes = sum(len(set(p_g[g_cv == g])) > 1 for g in np.unique(g_cv))
    print(f'pliegues por grupo: {np.bincount(p_g).tolist()} mensajes, {coupes} grupo(s) dividido(s) →',
          'correcto' if coupes == 0 and set(p_g) == {0, 1, 2} else 'revisar')
    Xe, Xv = standardiser(X_cv[p_g != 0], X_cv[p_g == 0])
    print('entrenamiento estandarizado: medias ≈ 0 y desviaciones típicas ≈ 1; validación: no exactamente →',
          'correcto' if np.allclose(Xe.mean(0), 0) and np.allclose(Xe.std(0), 1) and not np.allclose(Xv.mean(0), 0) else 'revisar')
    vc_groupe = {lam: score_vc(lam, p_g) for lam in LAMBDAS}
    lam_vc = min(vc_groupe, key=vc_groupe.get)
    print('VC por grupo:', {l: round(v, 3) for l, v in vc_groupe.items()}, '→ λ elegido:', lam_vc, '→',
          'correcto' if lam_vc == 0.01 else 'revisar')

In [ ]:
# d) Pliegues sorteados mensaje a mensaje (no hacerlo): la fuga de la lección 6.2, tres veces
if p_g is not None and score_vc(0.1, p_g) is not None:
    p_m = np.random.default_rng(1).integers(0, 3, len(y_cv))
    vc_message = {lam: score_vc(lam, p_m) for lam in LAMBDAS}
    print('VC mensaje a mensaje:', {l: round(v, 3) for l, v in vc_message.items()})
    plt.plot(range(5), list(vc_groupe.values()), 'o-', color=LOT_V, label='pliegues por grupo')
    plt.plot(range(5), list(vc_message.values()), 's--', color='grey', label='pliegues mensaje a mensaje (fuga)')
    plt.xticks(range(5), [str(l) for l in LAMBDAS]); plt.xlabel('λ'); plt.ylabel('Puntuación de validación cruzada')
    plt.legend(); plt.grid(alpha=0.4); plt.show()

    # e) Modelo final: estandarización y pesos sobre los 30 grupos, con el λ elegido; prueba UNA sola vez
    Xc, Xt = standardiser(X_cv, X_test)
    w_f, b_f = ajuster(Xc, y_cv, lam_vc)
    print(f'pérdida de prueba del modelo final (λ = {lam_vc}): {perte_moyenne(y_test, Xt, w_f, b_f):.3f}; '
          f'VC por grupo: {vc_groupe[lam_vc]:.3f}; VC mensaje a mensaje: {vc_message[lam_vc]:.3f}')

### d) y e) Solución comentada

**c)** Con pliegues por grupo, la puntuación de validación cruzada vale aproximadamente 0,547 para $\lambda = 0$, 0,493 para 0,001, **0,378 para 0,01**, 0,424 para 0,1 y 0,609 para 1. Baja y luego vuelve a subir: nos quedamos con $\lambda = 0{,}01$. Sin penalización, el filtro se sobreajusta a las seis características sin relación con la etiqueta; con $\lambda = 1$, los pesos quedan demasiado aplastados.

**d)** Con pliegues sorteados mensaje a mensaje, las puntuaciones bajan a 0,207, 0,206, 0,246, 0,385 y 0,598. Cada grupo queda dividido entre los pliegues: en cada vuelta, la validación contiene copias casi idénticas de mensajes de entrenamiento, y el filtro las reconoce. La puntuación es demasiado buena, sobre todo para los $\lambda$ pequeños, que premian el aprendizaje de memoria: esta división habría designado incluso $\lambda = 0{,}001$. La fuga falsea a la vez la medida y la elección.

**e)** El modelo final ($\lambda = 0{,}01$, estandarización y pesos calculados sobre los 30 grupos) tiene una pérdida de prueba de 0,356, medida una sola vez sobre 10 grupos nunca vistos. Está cerca de la puntuación de validación cruzada por grupo, 0,378, y lejos del 0,246 de los pliegues mensaje a mensaje. La validación cruzada por grupo anunciaba honestamente lo que le esperaba al filtro; la otra se equivocaba en un tercio. La diferencia restante entre 0,356 y 0,378 se debe al azar: 94 mensajes de prueba son pocos.

## Ejercicio 6.5 — Predecir y después comprobar: el recordatorio del teatro

*Conceptos: diferencia en puntos (ecuación 6.5), error estándar (ecuación 6.6), intervalo de confianza (ecuación 6.7).* Datos D6.E5. El vídeo solo merece la pena si hace ganar al menos 0,5 puntos.

### a) a e) Solución comentada

**a)** Tasa A: $126 / 1\,800 = 7{,}0\ \%$. Tasa B: $147 / 1\,750 = 8{,}4\ \%$. Diferencia: $8{,}4 - 7{,}0 = 1{,}4$ puntos porcentuales. Variación relativa: $1{,}4 / 7{,}0 = 20\ \%$.

**b)** En puntos: $7 \times 93 / 1\,800 \approx 0{,}362$ y $8{,}4 \times 91{,}6 / 1\,750 \approx 0{,}440$. $\text{ES} = \sqrt{0{,}362 + 0{,}440} = \sqrt{0{,}801} \approx 0{,}90$ punto. Intervalo: $1{,}4 \pm 1{,}96 \times 0{,}90 \approx 1{,}4 \pm 1{,}75$, es decir, de **−0,35 a 3,15 puntos** aproximadamente.

**c)** *El signo.* La diferencia medida es positiva, pero el cero está en el intervalo, e incluso pequeños efectos negativos: el ensayo no permite afirmar que el vídeo haya hecho reservar más. *La precisión.* El intervalo mide 3,5 puntos de ancho, dos veces y media la propia diferencia: el efecto se conoce muy mal. *El interés práctico.* El umbral de 0,5 puntos está en el intervalo, igual que el cero y efectos de 3 puntos: el vídeo puede ser muy útil o inútil; los datos no lo deciden. El colega confunde «el intervalo contiene el cero» con «el efecto es nulo»: un ensayo demasiado pequeño no demuestra la ausencia de efecto. 1,4 puntos sigue siendo el valor más plausible; lo que falta es precisión.

**d)** El error estándar varía como $1 / \sqrt{n}$: cuatro veces más espectadores dividen el error estándar, y por tanto la anchura del intervalo, por $\sqrt{4} = 2$, no por 4.

**e)** $\text{ES} = \sqrt{0{,}801 / 4} \approx 0{,}45$ punto, y el intervalo va de $1{,}4 - 0{,}88 \approx 0{,}52$ a $1{,}4 + 0{,}88 \approx 2{,}28$ puntos. El cero quedaría excluido, y el umbral de 0,5 puntos por muy poco: si las tasas siguieran siendo las mismas, el vídeo se consideraría útil, por poco.

In [ ]:
def intervalle(conv_A, n_A, conv_B, n_B):
    """Devuelve (diferencia, límite inferior, límite superior), en puntos porcentuales."""
    pA, pB = conv_A / n_A, conv_B / n_B
    es = np.sqrt(pA * (1 - pA) / n_A + pB * (1 - pB) / n_B)   # ecuación 6.6
    return 100 * (pB - pA), 100 * (pB - pA - 1.96 * es), 100 * (pB - pA + 1.96 * es)

In [ ]:
# Comprobación de a), b) y e)
A, B = theatre.iloc[0], theatre.iloc[1]
r = intervalle(A['reservations'], A['spectateurs'], B['reservations'], B['spectateurs'])
if r is None:
    print('aún sin completar. Completa la función.')
else:
    print(f'ensayo real: diferencia {r[0]:.2f} puntos, intervalo de {r[1]:.2f} a {r[2]:.2f} →',
          'correcto' if abs(r[0] - 1.4) < 1e-9 and abs(r[1] + 0.35) < 0.01 and abs(r[2] - 3.15) < 0.01 else 'revisar')
    r4 = intervalle(4 * A['reservations'], 4 * A['spectateurs'], 4 * B['reservations'], 4 * B['spectateurs'])
    print(f'ensayo cuatro veces más grande: de {r4[1]:.2f} a {r4[2]:.2f}; anchura {r[2] - r[1]:.2f} → {r4[2] - r4[1]:.2f} →',
          'correcto' if abs((r[2] - r[1]) / (r4[2] - r4[1]) - 2) < 1e-9 and r4[1] > 0.5 else 'revisar')

In [ ]:
# f) 1000 ensayos simulados cuyo efecto verdadero vale 1,4 puntos: ¿cuántos intervalos contienen 1,4?
rng = np.random.default_rng(65)
nA, nB, vrai_A, vrai_B = 1800, 1750, 0.07, 0.084
a = rng.binomial(nA, vrai_A, 1000) / nA
b = rng.binomial(nB, vrai_B, 1000) / nB
es_sim = np.sqrt(a * (1 - a) / nA + b * (1 - b) / nB)
contient = np.abs((b - a) - 0.014) <= 1.96 * es_sim
print(f'proporción de intervalos que contienen el efecto verdadero: {contient.mean():.1%}')
print(f'proporción de intervalos que contienen el cero: {np.mean(np.abs(b - a) <= 1.96 * es_sim):.1%}')

### f) Solución comentada

Alrededor del 94 % de los 1000 intervalos simulados contienen el efecto verdadero, 1,4 puntos: es el «95 %», salvo por el azar de la simulación. Califica el **método**: sobre un gran número de ensayos, alrededor del 95 % de los intervalos construidos así contienen el efecto verdadero. Observa también que, con un efecto verdadero de 1,4 puntos, una buena parte de los intervalos contienen el cero: un ensayo de este tamaño pasa a menudo por alto un efecto real. Es una razón más para no leer «contiene el cero» como «ningún efecto».

## Ejercicio 6.6 — Desafío: ¿cuántos clientes, y cuándo mirar?

*Conceptos: error estándar e intervalo de confianza (lección 6.6), distribución binomial (lección 4.5).*

### a) a c) Solución comentada

**a)** **Objetivo del dimensionamiento: la precisión.** Queremos que el intervalo al 95 % no se aleje más de $m = 1$ punto de la diferencia medida. No es un cálculo de potencia: no se fija ni una hipótesis que contrastar ni una probabilidad de detectar un efecto. **Hipótesis:** dos grupos asignados al azar del mismo tamaño $n$; tasas supuestas iguales a $p_A = 8$ % y $p_B = 10$ %, las del ensayo de la lección 6.5; aproximación por la curva de campana ($z = 1{,}96$); intervalo leído una sola vez, al final.

En puntos, $\text{ES} = \sqrt{\big(p_A(100 - p_A) + p_B(100 - p_B)\big) / n}$, y la condición $z \times \text{ES} \le m$ da:

$$n \ge \frac{z^2 \, \big(p_A(100 - p_A) + p_B(100 - p_B)\big)}{m^2} = \frac{1{,}96^2 \times (8 \times 92 + 10 \times 90)}{1^2} = 3{,}8416 \times 1\,636 \approx 6\,284{,}9 .$$

Se redondea al entero superior: **6285 clientes por grupo**, unos 12 570 en total. Estos 6285 clientes corresponden a un margen **aproximado** de un punto: aproximado porque la curva de campana es una aproximación, y porque el cálculo supone tasas del 8 % y del 10 %. Si las tasas reales difieren, el tamaño cambia: con tasas más cercanas al 50 %, harían falta más clientes.

**b)** El intervalo iría de $2{,}0 - 1{,}0 = 1{,}0$ a $3{,}0$ puntos: tocaría el umbral sin excluirlo. Para una semianchura de 0,5 puntos, hay que dividir el error estándar por 2 y, por tanto, multiplicar $n$ por 4: unos 25 140 clientes por grupo, es decir, 50 280 en total. Son más que los 40 000 suscriptores: un solo envío no bastaría; habría que repetir el ensayo, por ejemplo, en dos envíos.

**c)** No: se espera mucho más del 5 %, porque cada tarde da al azar una nueva ocasión de cruzar el límite.

In [ ]:
def taille_par_groupe(pA, pB, demi_largeur):
    """pA, pB en porcentaje; devuelve el menor n entero tal que 1,96 × EE ≤ demi_largeur."""
    return int(np.ceil((1.96 / demi_largeur) ** 2 * (pA * (100 - pA) + pB * (100 - pB))))

In [ ]:
# Comprobación de a) y b)
n1 = taille_par_groupe(8, 10, 1)
if n1 is None:
    print('aún sin completar. Completa la función.')
else:
    print(f'semianchura 1 punto: {n1} clientes por grupo →', 'correcto' if n1 == 6285 else 'revisar')
    print(f'semianchura 0,5 punto: {taille_par_groupe(8, 10, 0.5)} clientes por grupo, '
          f'{2 * taille_par_groupe(8, 10, 0.5)} en total')

In [ ]:
# d) 2000 ensayos SIN efecto. Calendario: 250 clientes por día y por grupo, 20 lecturas (una cada tarde,
#    sobre los clientes acumulados: 250, 500… 5000 por grupo). Regla de parada: primera tarde en que el intervalo excluye el 0.
rng = np.random.default_rng(66)
essais, jours, par_jour, p = 2000, 20, 250, 0.08
cA = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # pedidos acumulados, grupo A
cB = rng.binomial(par_jour, p, size=(essais, jours)).cumsum(axis=1)     # grupo B: misma tasa, ningún efecto
n = par_jour * np.arange(1, jours + 1)                                  # clientes por grupo al final de cada día
pA_, pB_ = cA / n, cB / n
es_j = np.sqrt(pA_ * (1 - pA_) / n + pB_ * (1 - pB_) / n)
exclut = np.abs(pB_ - pA_) > 1.96 * es_j                                # ¿excluye el cero el intervalo de la tarde?
fin, un_soir = exclut[:, -1].mean(), exclut.any(axis=1).mean()
print(f'intervalo leído solo el día 20: {fin:.1%} de los ensayos concluyen erróneamente que hay un efecto')
print(f'intervalo leído cada tarde, parada en el primer « éxito »: {un_soir:.1%}')

### d) y e) Solución comentada

Condiciones de la simulación: 2000 ensayos sin efecto (tasa del 8 % en ambos grupos), 250 clientes nuevos por día y por grupo, 20 lecturas como máximo, una cada tarde del día 1 al día 20, sobre los clientes acumulados desde el inicio (de 250 a 5000 por grupo); regla de parada: se para la primera tarde en que el intervalo al 95 % excluye el cero, por un lado o por el otro, y se concluye que hay un efecto. Leído una sola vez, la tarde del día 20, el intervalo excluye el cero en alrededor del 5 % de los ensayos sin efecto (4,6 % en la simulación): es el riesgo previsto. Con la lectura cada tarde y la regla de parada, se concluye erróneamente en alrededor de uno de cada cuatro ensayos (24,7 %). Esta cifra depende del calendario: más lecturas, o lecturas más seguidas, la harían subir todavía más. Cada tarde es una nueva ocasión para que el azar haga que el intervalo deje de contener el cero; pararse en ese momento es quedarse con las ocasiones favorables y tirar las demás. La regla: **fijar de antemano el tamaño del ensayo y la fecha de lectura**, por ejemplo 6300 clientes por grupo según a), y leer el intervalo en esa fecha. Mirar las cifras sobre la marcha no está prohibido, pero no debe decidir la parada.